# EEG augmentation (motor imagery)

Sliding windows, Gaussian noise, and light time warping on epochs from the Kaggle EDF dataset. Run cells in order.


## Install packages


In [1]:
!pip install mne scipy kagglehub matplotlib scikit-learn --quiet


DEPRECATION: Loading egg at /opt/miniconda3/lib/python3.12/site-packages/fermipy-1.3.1+21.g83b3-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330


## Imports


In [2]:
import re
import warnings
from collections import defaultdict
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import mne
import numpy as np
from scipy.interpolate import CubicSpline

warnings.filterwarnings("ignore")
mne.set_log_level("WARNING")
RNG = np.random.default_rng(42)
print("OK")


OK


## Download data


In [3]:
path = kagglehub.dataset_download("brianleung2020/eeg-motor-movementimagery-dataset")
dataset_path = Path(path)
edf_files = list(dataset_path.rglob("*.edf"))
print(len(edf_files), "plikow EDF")


Download already complete (2009033556 bytes).
Extracting files...


OSError: [Errno 28] No space left on device

## Load epochs

Adjust `N_SUBJECTS` if needed. Outputs `X` (epochs × channels × samples) and `y_labels`.


In [ ]:
MOTOR_EXEC_LR = [3, 7, 11]
MOTOR_EXEC_FF = [4, 8, 12]
IMAGERY_LR = [5, 9, 13]
IMAGERY_FF = [6, 10, 14]
ALL_TASK_RUNS = MOTOR_EXEC_LR + MOTOR_EXEC_FF + IMAGERY_LR + IMAGERY_FF


def parse_filename(filepath):
    m = re.match(r"S(\d+)R(\d+)", filepath.stem)
    return (int(m.group(1)), int(m.group(2))) if m else (None, None)


def get_run_event_mapping(run_num):
    if run_num in MOTOR_EXEC_LR + IMAGERY_LR:
        return {"T1": "Left Fist", "T2": "Right Fist"}
    if run_num in MOTOR_EXEC_FF + IMAGERY_FF:
        return {"T1": "Both Fists", "T2": "Both Feet"}
    return {}


def clean_channel_names(raw):
    mapping = {ch: ch.rstrip(".") for ch in raw.ch_names if ch.endswith(".")}
    if mapping:
        raw.rename_channels(mapping)
    montage = mne.channels.make_standard_montage("standard_1005")
    lower = {ch.lower(): ch for ch in montage.ch_names}
    case_map = {
        ch: lower[ch.lower()]
        for ch in raw.ch_names
        if ch.lower() in lower and lower[ch.lower()] != ch
    }
    if case_map:
        raw.rename_channels(case_map)
    try:
        raw.set_montage(montage, on_missing="ignore")
    except Exception:
        pass
    return raw


N_SUBJECTS = 8
TMIN, TMAX = -0.5, 4.0
L_FREQ, H_FREQ = 8.0, 30.0

subjects_files = defaultdict(list)
for f in edf_files:
    subj, run = parse_filename(f)
    if subj is not None and run in ALL_TASK_RUNS:
        subjects_files[subj].append((run, f))

subject_ids = sorted(subjects_files.keys())[:N_SUBJECTS]
epoch_list, label_list, sfreq = [], [], None

for subj_id in subject_ids:
    for run_num, filepath in subjects_files[subj_id]:
        ev_map = get_run_event_mapping(run_num)
        if not ev_map:
            continue
        try:
            raw = mne.io.read_raw_edf(filepath, preload=True, verbose=False)
            raw = clean_channel_names(raw)
            raw.filter(L_FREQ, H_FREQ, method="iir", verbose=False)
            if sfreq is None:
                sfreq = float(raw.info["sfreq"])
            events, event_dict = mne.events_from_annotations(raw, verbose=False)
            mapped = {
                name: event_dict[ann]
                for ann, name in ev_map.items()
                if ann in event_dict
            }
            if not mapped:
                continue
            epochs = mne.Epochs(
                raw,
                events,
                event_id=mapped,
                tmin=TMIN,
                tmax=TMAX,
                baseline=None,
                preload=True,
                verbose=False,
            )
            epochs.drop_bad(verbose=False)
            for cls_name in mapped:
                eps = epochs[cls_name]
                if len(eps) > 0:
                    epoch_list.append(eps.get_data())
                    label_list.extend([cls_name] * len(eps))
        except Exception:
            continue

X = np.concatenate(epoch_list, axis=0)
y_labels = np.array(label_list)
classes = np.unique(y_labels)
print("X:", X.shape, "sfreq:", sfreq, "Hz")
print("Klasy:", classes.tolist())
for c in classes:
    print(" ", c, int((y_labels == c).sum()))


## Augmentation functions

Sliding window shortens time; noise scales by per-channel std inside each epoch; time warp stretches/compresses time then resamples to the original length.


In [ ]:
def sliding_window_epochs(
    X,
    sfreq,
    window_sec=1.0,
    stride_sec=0.25,
):
    # Wycinanie nakladajacych sie okien z kazdej epoki.
    n_ep, n_ch, n_t = X.shape
    win = max(1, int(round(window_sec * sfreq)))
    step = max(1, int(round(stride_sec * sfreq)))
    if win > n_t:
        raise ValueError(f"window ({win} probek) > dlugosc epoki ({n_t})")
    outs, idx_map = [], []
    for i in range(n_ep):
        for start in range(0, n_t - win + 1, step):
            outs.append(X[i, :, start : start + win])
            idx_map.append(i)
    return np.stack(outs, axis=0), np.array(idx_map, dtype=int)


def add_gaussian_noise(X, noise_std_scale=0.05, rng=None):
    # Szum: std = noise_std_scale * std(kanal) w obrebie epoki.
    rng = rng or np.random.default_rng()
    out = np.empty_like(X, dtype=np.float64)
    for i in range(X.shape[0]):
        x = X[i].astype(np.float64, copy=False)
        ch_std = np.std(x, axis=1, keepdims=True)
        ch_std = np.where(ch_std < 1e-12, 1.0, ch_std)
        noise = rng.normal(0.0, 1.0, size=x.shape).astype(np.float64)
        out[i] = x + noise_std_scale * ch_std * noise
    return out.astype(X.dtype, copy=False)


def time_warp_epochs(X, scale_range=(0.92, 1.08), rng=None):
    # Lekkie rozciagniecie/scisniecie czasu; wspolna deformacja dla kanalow w epoce.
    rng = rng or np.random.default_rng()
    n_ep, n_ch, n_t = X.shape
    t_old = np.linspace(0.0, 1.0, n_t)
    out = np.empty_like(X, dtype=np.float64)
    for i in range(n_ep):
        scale = float(rng.uniform(*scale_range))
        n_new = max(2, int(round(n_t * scale)))
        t_new_axis = np.linspace(0.0, 1.0, n_new)
        warped = np.empty((n_ch, n_new), dtype=np.float64)
        for c in range(n_ch):
            spline = CubicSpline(t_old, X[i, c].astype(np.float64))
            warped[c] = spline(t_new_axis)
        t_res = np.linspace(0.0, 1.0, n_t)
        for c in range(n_ch):
            out[i, c] = np.interp(t_res, t_new_axis, warped[c])
    return out.astype(X.dtype, copy=False)


## Sliding window example

Compare one full epoch to the first 1 s crop (C3 if available).


In [ ]:
ch_demo = "C3"
# Indeks kanalu (pierwszy plik / pierwsza epoka — nazwy jak w danych)
# Szukamy C3 w danych zaladowanych w kroku 4 — potrzebujemy listy nazw z jednego raw
sample_path = subjects_files[subject_ids[0]][0][1]
raw0 = clean_channel_names(
    mne.io.read_raw_edf(sample_path, preload=True, verbose=False)
)
if ch_demo in raw0.ch_names:
    ich = raw0.ch_names.index(ch_demo)
else:
    ich = 0
    ch_demo = raw0.ch_names[0]

win_sec, stride_sec = 1.0, 0.25
X_sw, src_idx = sliding_window_epochs(X, sfreq, win_sec, stride_sec)
y_sw = y_labels[src_idx]

print(f"Oryginal: {X.shape[0]} epok -> sliding: {X_sw.shape[0]} okien")
print(f"Ksztalt okna: {X_sw.shape[1]} kan x {X_sw.shape[2]} probek "
      f"({X_sw.shape[2] / sfreq:.2f} s)")

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=False)
t_full = np.arange(X.shape[2]) / sfreq + TMIN
axes[0].plot(t_full, X[0, ich] * 1e6, lw=0.8)
axes[0].set_title(f"Pierwsza epoka, kanal {ch_demo} (pelne okno)")
axes[0].set_ylabel("uV")
win_samples = X_sw.shape[2]
t_win = np.arange(win_samples) / sfreq + TMIN
axes[1].plot(t_win, X_sw[0, ich] * 1e6, lw=0.8)
axes[1].set_title("Pierwsze wyciete okno sliding (1 s)")
axes[1].set_xlabel("czas (s)")
axes[1].set_ylabel("uV")
plt.tight_layout()
plt.show()


## Noise and time warp

Same epoch and channel: original, noisy, and warped.


In [ ]:
ep = 0
x0 = X[ep : ep + 1]
x_noisy = add_gaussian_noise(x0, noise_std_scale=0.08, rng=RNG)
x_warp = time_warp_epochs(x0, scale_range=(0.94, 1.06), rng=RNG)

t = np.arange(X.shape[2]) / sfreq + TMIN
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t, x0[0, ich] * 1e6, label="oryginal", alpha=0.9)
ax.plot(t, x_noisy[0, ich] * 1e6, label="+ szum Gaussa", alpha=0.7)
ax.plot(t, x_warp[0, ich] * 1e6, label="time warp", alpha=0.7)
ax.set_xlabel("czas (s)")
ax.set_ylabel("uV")
ax.set_title(f"Epoka {ep}, kanal {ch_demo}")
ax.legend()
plt.tight_layout()
plt.show()


## Batch helper

Only augment the training split after train/test split so labels do not leak into validation or test.


In [ ]:
def build_augmented_batch(
    X_batch,
    y_batch,
    sfreq,
    n_noise_copies=2,
    noise_std_scale=0.06,
    apply_warp_prob=0.5,
    rng=None,
):
    # Oryginal + kopie z szumem + opcjonalnie czesc epok z time warp.
    rng = rng or np.random.default_rng()
    X_list = [X_batch]
    y_list = [y_batch]
    for _ in range(n_noise_copies):
        X_list.append(add_gaussian_noise(X_batch, noise_std_scale, rng))
        y_list.append(y_batch.copy())
    if apply_warp_prob > 0:
        mask = rng.random(len(X_batch)) < apply_warp_prob
        if mask.any():
            Xw = X_batch.copy()
            idx = np.where(mask)[0]
            Xw[idx] = time_warp_epochs(X_batch[idx], rng=rng)
            X_list.append(Xw)
            y_list.append(y_batch.copy())
    return np.concatenate(X_list, axis=0), np.concatenate(y_list, axis=0)


# Mini-demonstracja liczebnosci
X_demo, y_demo = build_augmented_batch(
    X[:50], y_labels[:50], sfreq, n_noise_copies=2, apply_warp_prob=0.5, rng=RNG
)
print("Batch 50 epok -> po augmentacji:", X_demo.shape[0], "probek")


## Optional: CSP + LDA

5-fold CV with augmentation on the train fold only. Needs `scikit-learn` in the environment.


In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from mne.decoding import CSP

le = LabelEncoder()
y_enc = le.fit_transform(y_labels)
n_comp = min(6, X.shape[1], len(np.unique(y_enc)))

clf_base = Pipeline(
    [
        ("csp", CSP(n_components=n_comp, reg=None, log=True, norm_trace=False)),
        ("scaler", StandardScaler()),
        ("lda", LinearDiscriminantAnalysis()),
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores_no_aug = cross_val_score(clf_base, X, y_enc, cv=cv, scoring="accuracy", n_jobs=-1)

rng_aug = np.random.default_rng(123)


def augment_train(X_tr, y_tr):
    Xa, ya = build_augmented_batch(
        X_tr,
        y_tr,
        sfreq,
        n_noise_copies=2,
        noise_std_scale=0.06,
        apply_warp_prob=0.4,
        rng=rng_aug,
    )
    return Xa, ya


scores_aug = []
for train_i, test_i in cv.split(X, y_enc):
    X_tr, y_tr = augment_train(X[train_i], y_enc[train_i])
    pipe = Pipeline(
        [
            ("csp", CSP(n_components=n_comp, reg=None, log=True, norm_trace=False)),
            ("scaler", StandardScaler()),
            ("lda", LinearDiscriminantAnalysis()),
        ]
    )
    pipe.fit(X_tr, y_tr)
    scores_aug.append(pipe.score(X[test_i], y_enc[test_i]))
scores_aug = np.array(scores_aug)

print("LDA+CSP accuracy (bez aug):", scores_no_aug.mean(), "+/-", scores_no_aug.std())
print("LDA+CSP accuracy (aug na train):", scores_aug.mean(), "+/-", scores_aug.std())
